In [ ]:
# prompt: mount drive

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Conv2DTranspose, concatenate, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import img_to_array, load_img
from skimage.metrics import structural_similarity as compare_ssim
from skimage.metrics import peak_signal_noise_ratio as compare_psnr
from tqdm import tqdm

# Directories for images (update these paths if necessary)
colon_aca_path = "/content/drive/MyDrive/archive (1)/lung_colon_image_set/colon_image_sets/colon_aca"
colon_n_path = "/content/drive/MyDrive/archive (1)/lung_colon_image_set/colon_image_sets/colon_n"
lung_aca_path = "/content/drive/MyDrive/archive (1)/lung_colon_image_set/lung_image_sets/lung_aca"
lung_n_path = "/content/drive/MyDrive/archive (1)/lung_colon_image_set/lung_image_sets/lung_n"
lung_scc_path = "/content/drive/MyDrive/archive (1)/lung_colon_image_set/lung_image_sets/lung_scc"

# Define output directory in Google Drive
output_dir = "/content/drive/MyDrive/denoised_images"  # Update with your preferred folder path

# Create output directory if it doesn't exist
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Function to load and process images (for RGB)
def load_and_preprocess_images(image_paths, img_size_target, n_images=300):
    images = []
    image_filenames = os.listdir(image_paths)[:n_images]  # Select only first n_images images
    for filename in tqdm(image_filenames):
        img_path = os.path.join(image_paths, filename)
        img = load_img(img_path, target_size=(img_size_target, img_size_target), color_mode='rgb')  # Load as RGB
        img = img_to_array(img) / 255.0  # Normalize to [0,1]
        images.append(img)
    return np.array(images), image_filenames  # Return images and their filenames


In [ ]:

# Build U-Net model
def build_model(input_layer, start_neurons):
    # Encoder
    conv1 = Conv2D(start_neurons*1, (3,3), activation='relu', padding='same')(input_layer)
    conv1 = Conv2D(start_neurons*1, (3,3), activation='relu', padding='same')(conv1)
    pool1 = MaxPooling2D((2,2))(conv1)
    pool1 = Dropout(0.25)(pool1)

    conv2 = Conv2D(start_neurons*2, (3,3), activation='relu', padding='same')(pool1)
    conv2 = Conv2D(start_neurons*2, (3,3), activation='relu', padding='same')(conv2)
    pool2 = MaxPooling2D((2,2))(conv2)
    pool2 = Dropout(0.5)(pool2)

    conv3 = Conv2D(start_neurons*4, (3,3), activation='relu', padding='same')(pool2)
    conv3 = Conv2D(start_neurons*4, (3,3), activation='relu', padding='same')(conv3)
    pool3 = MaxPooling2D((2,2))(conv3)
    pool3 = Dropout(0.5)(pool3)

    conv4 = Conv2D(start_neurons*8, (3,3), activation='relu', padding='same')(pool3)
    conv4 = Conv2D(start_neurons*8, (3,3), activation='relu', padding='same')(conv4)
    pool4 = MaxPooling2D((2,2))(conv4)
    pool4 = Dropout(0.5)(pool4)

    # Bottleneck
    convm = Conv2D(start_neurons*16, (3,3), activation='relu', padding='same')(pool4)
    convm = Conv2D(start_neurons*16, (3,3), activation='relu', padding='same')(convm)

    # Decoder
    deconv4 = Conv2DTranspose(start_neurons*8, (3,3), strides=(2,2), padding='same')(convm)
    uconv4 = concatenate([deconv4, conv4])
    uconv4 = Dropout(0.5)(uconv4)
    uconv4 = Conv2D(start_neurons*8, (3,3), activation='relu', padding='same')(uconv4)
    uconv4 = Conv2D(start_neurons*8, (3,3), activation='relu', padding='same')(uconv4)

    deconv3 = Conv2DTranspose(start_neurons*4, (3,3), strides=(2,2), padding='same')(uconv4)
    uconv3 = concatenate([deconv3, conv3])
    uconv3 = Dropout(0.5)(uconv3)
    uconv3 = Conv2D(start_neurons*4, (3,3), activation='relu', padding='same')(uconv3)
    uconv3 = Conv2D(start_neurons*4, (3,3), activation='relu', padding='same')(uconv3)

    deconv2 = Conv2DTranspose(start_neurons*2, (3,3), strides=(2,2), padding='same')(uconv3)
    uconv2 = concatenate([deconv2, conv2])
    uconv2 = Dropout(0.5)(uconv2)
    uconv2 = Conv2D(start_neurons*2, (3,3), activation='relu', padding='same')(uconv2)
    uconv2 = Conv2D(start_neurons*2, (3,3), activation='relu', padding='same')(uconv2)

    deconv1 = Conv2DTranspose(start_neurons*1, (3,3), strides=(2,2), padding='same')(uconv2)
    uconv1 = concatenate([deconv1, conv1])
    uconv1 = Dropout(0.5)(uconv1)
    uconv1 = Conv2D(start_neurons*1, (3,3), activation='relu', padding='same')(uconv1)
    uconv1 = Conv2D(start_neurons*1, (3,3), activation='relu', padding='same')(uconv1)

    output_layer = Conv2D(3, (1,1), padding='same', activation='sigmoid')(uconv1)  # 3 channels for RGB
    return output_layer


In [ ]:
# Define target image size and input size (3 channels for RGB)
img_size_target = 48
input_layer = Input((img_size_target, img_size_target, 3))  # 3 channels for RGB
output_layer = build_model(input_layer, 64)

# Initializing and compiling the model
model_unet = Model(input_layer, output_layer)
model_unet.compile(optimizer='adam', loss='MSE')

# Loading and processing images
colon_aca_images, colon_aca_filenames = load_and_preprocess_images(colon_aca_path, img_size_target)
colon_n_images, colon_n_filenames = load_and_preprocess_images(colon_n_path, img_size_target)
lung_aca_images, lung_aca_filenames = load_and_preprocess_images(lung_aca_path, img_size_target)
lung_n_images, lung_n_filenames = load_and_preprocess_images(lung_n_path, img_size_target)
lung_scc_images, lung_scc_filenames = load_and_preprocess_images(lung_scc_path, img_size_target)

# Combine all images
x_train_noisy = np.vstack((colon_aca_images, colon_n_images, lung_aca_images, lung_n_images, lung_scc_images))
x_train = x_train_noisy.copy()  # Assuming the noisy images are the same as the clean ones

# Train the model
model_unet.fit(x_train_noisy, x_train, epochs=10, batch_size=32, shuffle=True)

# Predict denoised images
denoised_images = model_unet.predict(x_train_noisy)

100%|██████████| 300/300 [03:32<00:00,  1.41it/s]


Epoch 1/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 603s 13s/step - loss: 0.0431
Epoch 2/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 646s 13s/step - loss: 0.0113
Epoch 3/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 624s 13s/step - loss: 0.0056
Epoch 4/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 585s 12s/step - loss: 0.0038
Epoch 5/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 622s 12s/step - loss: 0.0030
Epoch 6/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 632s 13s/step - loss: 0.0026
Epoch 7/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 583s 12s/step - loss: 0.0026
Epoch 8/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 618s 12s/step - loss: 0.0021
Epoch 9/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 572s 12s/step - loss: 0.0024
Epoch 10/10
47/47 ━━━━━━━━━━━━━━━━━━━━ 623s 12s/step - loss: 0.0019
47/47 ━━━━━━━━━━━━━━━━━━━━ 132s 3s/step


In [ ]:
# Save the denoised images to the output folder in Google Drive
def save_denoised_images(output_dir, denoised_images, original_filenames):
    for idx, img in enumerate(denoised_images):
        # Convert image back to [0, 255] range
        img = (img * 255).astype(np.uint8)
        img_filename = original_filenames[idx]
        img_output_path = os.path.join(output_dir, f"denoised_{img_filename}")
        # Save the image in RGB format
        cv2.imwrite(img_output_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))  # Convert RGB to BGR for OpenCV

# Save images to Google Drive
# Keep track of the indices for slicing the denoised_images array
num_images = 300  # Number of images per category

save_denoised_images(output_dir, denoised_images[0:num_images], colon_aca_filenames)
save_denoised_images(output_dir, denoised_images[num_images:2*num_images], colon_n_filenames)
save_denoised_images(output_dir, denoised_images[2*num_images:3*num_images], lung_aca_filenames)
save_denoised_images(output_dir, denoised_images[3*num_images:4*num_images], lung_n_filenames)
save_denoised_images(output_dir, denoised_images[4*num_images:5*num_images], lung_scc_filenames)  # Assuming lung_scc is the 5th category


In [ ]:
# Evaluate the denoised images
def evaluate_images(denoised_images, original_images):
    ssim_scores = []
    psnr_scores = []

    for i in range(len(denoised_images)):
        # Ensure images are in range [0, 255] for evaluation
        denoised_img = (denoised_images[i] * 255).astype(np.uint8)
        original_img = (original_images[i] * 255).astype(np.uint8)

        # Convert images to grayscale for SSIM and PSNR calculation
        denoised_img_gray = cv2.cvtColor(denoised_img, cv2.COLOR_RGB2GRAY)
        original_img_gray = cv2.cvtColor(original_img, cv2.COLOR_RGB2GRAY)

        # Calculate SSIM and PSNR
        ssim_score = compare_ssim(original_img_gray, denoised_img_gray)
        psnr_score = compare_psnr(original_img_gray, denoised_img_gray)

        ssim_scores.append(ssim_score)
        psnr_scores.append(psnr_score)

    # Compute average SSIM and PSNR
    avg_ssim = np.mean(ssim_scores)
    avg_psnr = np.mean(psnr_scores)

    print(f"Average SSIM: {avg_ssim:.4f}")
    print(f"Average PSNR: {avg_psnr:.2f} dB")

# Evaluate the model
evaluate_images(denoised_images, x_train)

Average SSIM: 0.9932
Average PSNR: 35.79 dB
